# Learning Objectives

In this notebook, you will craft sophisticated ETL jobs that interface with a variety of common data sources, such as 
- REST APIs (HTTP endpoints)
- RDBMS
- Hive tables (managed tables)
- Various file formats (csv, json, parquet, etc.)


# Interview Questions

As you progress through the practice, attempt to answer the following questions:

## Columnar File
- What is a columnar file format and what advantages does it offer?
- Why is Parquet frequently used with Spark and how does it function?
- How do you read/write data from/to a Parquet file using a DataFrame?

## Partitions
- How do you save data to a file system by partitions? (Hint: Provide the code)
- How and why can partitions reduce query execution time? (Hint: Give an example)

## JDBC and RDBMS
- How do you load data from an RDBMS into Spark? (Hint: Discuss the steps and JDBC)

## REST API and HTTP Requests
- How can Spark be used to fetch data from a REST API? (Hint: Discuss making API requests)

## ETL Job One: Parquet file
### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Data transformation requirements https://pgexercises.com/questions/aggregates/fachoursbymonth.html

### Load
Load data into a parquet file

### What is Parquet? 

Columnar files are an important technique for optimizing Spark queries. Additionally, they are often tested in interviews.
- https://www.youtube.com/watch?v=KLFadWdomyI
- https://www.databricks.com/glossary/what-is-parquet

In [0]:
%sql

USE CATALOG 'jarvis-catalog';
USE SCHEMA bronze;


In [0]:
bookings = spark.sql("select * from bookings")
bookings.show(5)

+------+-----+-----+-------------------+-----+
|bookid|facid|memid|          starttime|slots|
+------+-----+-----+-------------------+-----+
|     0|    3|    1|2012-07-03 11:00:00|    2|
|     1|    4|    1|2012-07-03 08:00:00|    2|
|     2|    6|    0|2012-07-03 18:00:00|    2|
|     3|    7|    1|2012-07-03 19:00:00|    2|
|     4|    8|    1|2012-07-03 10:00:00|    1|
+------+-----+-----+-------------------+-----+
only showing top 5 rows


In [0]:

bookings = spark.sql("select * from bookings")
members = spark.sql("select * from members")
facilities = spark.sql("select * from facilities")

In [0]:
import pyspark.sql.functions as F

In [0]:
# Write your solution here
transformations = bookings \
    .filter(bookings.starttime >= '2012-09-01') \
    .filter(bookings.starttime < '2012-10-01') \
    .groupby(bookings.facid) \
    .agg(F.sum(bookings.slots).alias("total_slots")) \
    .select(bookings.facid.alias("facid"), "total_slots") 


write_location = "/Volumes/jarvis-catalog/silver/data"

file_name = "Etl_Job_One.parquet"
transformations.write.mode('overwrite').parquet(f"{write_location}/{file_name}")



## ETL Job Two: Partitions

### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Transform the data https://pgexercises.com/questions/joins/threejoin.html

### Load
Partition the result data by facility column and then save to `threejoin_delta` managed table. Additionally, they are often tested in interviews.

hint: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html

What are paritions? 

Partitions are an important technique to optimize Spark queries
- https://www.youtube.com/watch?v=hvF7tY2-L3U&t=268s

In [0]:
# Write your solution here
bookings_two = spark.sql("select * from bookings")
members_two = spark.sql("select * from members")
facilities_two = spark.sql("select * from facilities")

In [0]:
transformations_two = members_two \
    .join(bookings_two, members_two.memid == bookings_two.memid, "inner") \
    .join(facilities_two, bookings_two.facid == facilities_two.facid, "inner") \
    .filter((facilities_two.name == "Tennis Court 2") | (facilities_two.name == "Tennis Court 1")) \
    .withColumn("full_name", F.concat(members_two.firstname, F.lit(" "), members_two.surname)) \
    .select("full_name", facilities_two.name.alias("facility_name")) \
    .distinct() \
    .orderBy("full_name", "facility_name")

transformations_two.write.mode('overwrite').partitionBy('facility_name').saveAsTable('`jarvis-catalog`.silver.threejoin_delta')

## ETL Job Three: HTTP Requests

### Extract
Extract daily stock price data price from the following companies, Google, Apple, Microsoft, and Tesla. 

Data Source
- API: https://rapidapi.com/alphavantage/api/alpha-vantage
- Endpoint: GET `TIME_SERIES_DAILY`

Sample HTTP request

```
curl --request GET \
	--url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=TSLA&outputsize=compact&datatype=json' \
	--header 'X-RapidAPI-Host: alpha-vantage.p.rapidapi.com' \
	--header 'X-RapidAPI-Key: [YOUR_KEY]'

```

Sample Python HTTP request

```
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":"IBM",
    "datatype":"json",
    "outputsize":"compact"
}

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

response = requests.get(url, headers=headers, params=querystring)

data = response.json()

# Now 'data' contains the daily time series data for "IBM"
```

### Transform
Find **weekly** max closing price for each company.

hints: 
  - Use a `for-loop` to get stock data for each company
  - Use the spark `union` operation to concat all data into one DF
  - create a new `week` column from the data column
  - use `group by` to calcualte max closing price

### Load
- Partition `DF` by company
- Load the DF in to a managed table called, `max_closing_price_weekly`

In [0]:
import requests
from functools import reduce
from pyspark.sql import DataFrame
import time

In [0]:


url = "https://alpha-vantage.p.rapidapi.com/query"
headers = {
    'x-rapidapi-key': "0dd0550f3cmshc01bcb41ee0fcbap1262e6jsn3dc6b85248d5",
    'x-rapidapi-host': "alpha-vantage.p.rapidapi.com",
}

     

def getStockData(company):
    querystring = {
        "function": "TIME_SERIES_DAILY",
        "symbol": f"{company}",
        "datatype": "json",
        "outputsize": "compact"
    }
    
    #ensure that the request is successful and handle any potential errors
    try:
        response = requests.get(url, headers=headers, params=querystring, timeout=6)
        response.raise_for_status()
        payload = response.json()
        # extract the time series
        data = payload["Time Series (Daily)"]
    #generic fails 
    except requests.exceptions.RequestException as e:
        print(f"Request failed for {company}: {e}")
        return None
    #handle missings
    except KeyError:
        print(f"Unexpected response for {company}: {payload}")
        return None
    
    # create a DataFrame from the time series data
    df = spark.createDataFrame([{"timeseries": data}])
    df = df.select(F.explode("timeseries").alias("date", "metrics"))

    company_df = (df
        .select(
            "date",
            F.col("metrics")["1. open"].cast("double").alias("open"),
            F.col("metrics")["2. high"].cast("double").alias("high"),
            F.col("metrics")["3. low"].cast("double").alias("low"),
            F.col("metrics")["4. close"].cast("double").alias("close"),
            F.col("metrics")["5. volume"].cast("long").alias("volume"),
        )
        .withColumn("symbol", F.lit(f"{company}"))
    )
       
       
    # date comes back as a string, cast it to a real date
    company_df = company_df.withColumn("date", F.col("date").cast("date"))
    return company_df


companies = ['GOOGL', 'AAPL', 'MSFT', 'TSLA']
# companies = ['GOOGL']
requests_data = []
for company in companies:
        requests_data.append(getStockData(company))
        time.sleep(15)
    
dfs = [df for df in requests_data]
stock_price_df = reduce(DataFrame.union, dfs)
display(stock_price_df.tail(5))


date,open,high,low,close,volume,symbol
2026-02-23,407.285,407.7,394.04,399.83,69680026,TSLA
2026-02-20,408.3,414.7,405.5,411.82,57912225,TSLA
2026-02-19,407.25,415.2499,404.11,411.71,51019638,TSLA
2026-02-18,411.11,416.9,409.58,411.32,45921402,TSLA
2026-02-17,412.36,413.72,400.51,410.63,59678789,TSLA


## ETL Job Four: RDBMS


### Extract
Extract RNA data from a public PostgreSQL database.

- https://rnacentral.org/help/public-database
- Extract 100 RNA records from the `rna` table (hint: use `limit` in your sql)
- hint: use `spark.read.jdbc` https://docs.databricks.com/external-data/jdbc.html

### Transform
We want to load the data as it so there is no transformation required.


### Load
Load the DF in to a managed table called, `rna_100_records`

In [0]:
table_name = "rna"
db_URL = "hh-pgsql-public.ebi.ac.uk"
port = 5432
db_name = "pfmegrnargs"
username = "reader"
password = "NWDMCE5xdipIjRrp"

rna_df_100 = (spark.read
  .format("jdbc")
  .option("url", f"jdbc:postgresql://{db_URL}:{port}/{db_name}")
  .option("dbtable", f"(SELECT * FROM {table_name} LIMIT 100) AS t")
  .option("user", username)
  .option("password", password)
  .load()
)

rna_df_100.show()

# rna_df_100.write.mode("overwrite").saveAsTable("rna_100_records")
     

+--------+-------------+--------------------+---------+----------------+----+--------------------+--------+--------------------+
|      id|          upi|           timestamp|userstamp|           crc64| len|           seq_short|seq_long|                 md5|
+--------+-------------+--------------------+---------+----------------+----+--------------------+--------+--------------------+
|34093374|URS000208393E|2020-08-19 23:22:...|   rnacen|8025DC31F1104AD2| 292|GTGCCAGCAGCCTCGGT...|    NULL|e1139b319ded12572...|
|34093375|URS000208393F|2020-08-19 23:22:...|   rnacen|A3EAC43AF237B7D9| 308|GATGAACGCTAGCGGCA...|    NULL|e113a03a0f15e241a...|
|34093376|URS0002083940|2020-08-19 23:22:...|   rnacen|C0678A96A8DB0D3C| 437|GATGAACGCTGGCGGCA...|    NULL|e113a0655393db9a9...|
|34093377|URS0002083941|2020-08-19 23:22:...|   rnacen|BE37C733EEA8C1C8| 291|GTGTCAGCCGCCGCGGT...|    NULL|e113a1abf4c146194...|
|34093378|URS0002083942|2020-08-19 23:22:...|   rnacen|1FF9D97E0610D5E0| 450|CCTACGGGGGGCAGCAG...